In [0]:
from pyspark.sql.types import *
from datetime import datetime, timedelta
from pyspark.sql.functions import *
from pyspark.sql.types import *
import random
from pyspark.sql.window import Window

In [0]:


snapshot_rows = []

base_time = datetime(2024, 1, 1, 9, 0)

for i in range(1, 15001):
    snapshot_rows.append((
        f"O{i}",
        f"C{i}",
        random.choice(["CREATED", "UPDATED", "COMPLETED"]),
        round(random.uniform(100, 5000), 2),
        base_time + timedelta(days=0)
    ))

snapshot_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("order_amount", DoubleType(), True),
    StructField("last_updated_time", TimestampType(), False)
])

orders_snapshot_df = spark.createDataFrame(snapshot_rows, snapshot_schema)


In [0]:
increment_rows = []

for i in range(5000, 20001):
    increment_rows.append((
        f"O{i}",
        f"C{i}",
        random.choice(["UPDATED", "CANCELLED"]),
        None if random.random() < 0.3 else round(random.uniform(100, 5000), 2),
        base_time + timedelta(days=1)
    ))

increment_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("order_amount", DoubleType(), True),
    StructField("event_time", TimestampType(), False)
])

orders_increment_df = spark.createDataFrame(increment_rows, increment_schema)


In [0]:
display(orders_snapshot_df)
display(orders_increment_df)

In [0]:
filter_snap = orders_snapshot_df.filter(col("order_id") == "O5000")
display(filter_snap)
filter_incr = orders_increment_df.filter(col("order_id") == "O5000")
display(filter_incr)

In [0]:
snapshot = orders_snapshot_df.withColumn("tb_flag", lit(2))
increment = orders_increment_df.withColumn("tb_flag", lit(1))

mergetable = snapshot.union(increment)

In [0]:
filter_merge = mergetable.filter(col("order_id") == "O5002")
display(filter_merge)

In [0]:
window = Window.partitionBy("order_id") \
            .orderBy(col("last_updated_time").desc()) \
            .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

row_filter = Window.partitionBy("order_id") \
                .orderBy(col("tb_flag"), col("last_updated_time").desc())

window_df = mergetable.withColumn("final_order_amount", last("order_amount", ignorenulls = True).over(window)) \
                    .withColumn("row_filter", row_number().over(row_filter)) \
                    .filter(col("row_filter") == 1) \
                    .select(
                        col("order_id"),
                        col("customer_id"),
                        col("order_status"),
                        col("final_order_amount"),
                        col("last_updated_time"),
                        col("tb_flag")
                    ) \
                    .orderBy("order_id")

display(window_df)

### Working with Delta Table

from delta.tables import DeltaTable

delta_tbl = DeltaTable.forPath(spark, "/path/to/orders")

delta_tbl.alias("t").merge(
    updates_df.alias("s"),
    "t.order_id = s.order_id"
).whenMatchedUpdate(set={
    "status": "s.status",
    "amount": "coalesce(s.amount, t.amount)"
}).whenNotMatchedInsertAll().execute()